# E. coli iML1515 MINN Table 2 Benchmarks

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HFTOKEN")

In [2]:
from datetime import date

import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

from train_flux_transformer import load_data, prepare_tensors, set_seed

%matplotlib inline
pd.set_option('display.max_rows', None)

In [3]:
class AttentionBlock(nn.Module):
    """Custom multi-head attention block for metabolic modeling"""
    def __init__(self, d_model=128, n_heads=8, dropout=0.05):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.layer_norm = nn.LayerNorm(d_model)

        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.head_scores = nn.Parameter(torch.zeros(n_heads))

    def forward(self, x, c):
        # x: (batch, seq_len, d_model)
        # c: (batch, seq_len, 1)
        x_norm = self.layer_norm(x) # pre-norm
        attn_out, attn_weights = self.mha(x_norm, x_norm, x_norm, need_weights=True, average_attn_weights=False)

        x_out = attn_out + x

        # Per-head diffusion of c:
        c_heads = torch.matmul(attn_weights, c.unsqueeze(1))
        alpha = F.softmax(self.head_scores, dim=0).view(1, self.n_heads, 1, 1)  # (1,H,1,1)
        c_att = (c_heads * alpha).sum(dim=1)  # (B, S, 1)

        c_out = c_att + c

        return x_out, c_out

In [4]:
class FeedForwardBlock(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.05):
        super().__init__()

        self.d_model = d_model + 1
        self.d_ff = d_ff

        self.layer_norm = nn.LayerNorm(self.d_model)
        self.linear1 = nn.Linear(self.d_model, self.d_ff)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(self.d_ff, self.d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, c):
        y = torch.cat((x, c), dim=2)
        
        norm_y = self.layer_norm(y)
        hidden = self.linear1(norm_y)
        hidden = self.activation(hidden)
        hidden = self.dropout(hidden)
        output = self.linear2(hidden)

        return output + y

In [5]:
class FluxTransformerLayer(nn.Module):
    """Single transformer block without embedding layer"""
    def __init__(self, d_model=128, n_heads=8, d_ff=1024, dropout=0.05):
        super().__init__()
        self.d_model = d_model
        
        self.attention_block = AttentionBlock(d_model, n_heads, dropout)
        self.feedforward_block = FeedForwardBlock(d_model, d_ff, dropout)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
    
    def forward(self, x, c):
        attn_x, attn_c = self.attention_block(x, c)
        ff_output = self.feedforward_block(attn_x, attn_c)
        
        # Split the concatenated output
        updated_x = ff_output[:, :, :-1]
        updated_c = ff_output[:, :, -1:]
        
        return updated_x, updated_c
        
class FluxTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        input_token_indices,          # list[int] or 1D tensor[int64]
        d_model=128,
        n_heads=8,
        n_layers=3,
        d_ff=1024,
        dropout=0.05,
    ):
        super().__init__()
        if vocab_size is None:
            raise ValueError("vocab_size must be provided explicitly.")
        self.vocab_size = int(vocab_size)
        self.d_model = d_model

        idx = torch.as_tensor(input_token_indices, dtype=torch.long)
        if idx.ndim != 1:
            raise ValueError("input_token_indices must be 1D.")
        if (idx < 0).any() or (idx >= self.vocab_size).any():
            raise ValueError("input_token_indices contains out-of-range indices.")
        # Register as buffer so it moves with .to(device)
        self.register_buffer("input_token_indices", idx, persistent=True)

        self.input_embedding = nn.Embedding(self.vocab_size, d_model)

        self.layers = nn.ModuleList([
            FluxTransformerLayer(d_model=d_model, n_heads=n_heads, d_ff=d_ff, dropout=dropout)
            for _ in range(n_layers)
        ])

    def forward(self, c, output_subset=None, return_embedding=False):
        """
        c: (batch, vocab_size, 1)
        output_subset: 1D tensor of token indices to train on (typically excludes injected tokens)
        """
        batch_size = c.size(0)

        always = self.input_token_indices  # (n_injected,)

        if output_subset is None:
            selected_indices = torch.arange(self.vocab_size, device=c.device)
        else:
            output_subset = output_subset.to(c.device).long()
            selected_indices = torch.unique(torch.cat([always, output_subset]), sorted=True)

        y = selected_indices.unsqueeze(0).expand(batch_size, -1)      # (B, S)
        x = self.input_embedding(y)                                    # (B, S, d_model)

        c_subset = c[:, selected_indices, :]                           # (B, S, 1)
        c_subset_all_layers = torch.zeros(batch_size, c_subset.size(1), len(self.layers), device=c.device)

        for e, layer in enumerate(self.layers):
            x, c_subset = layer(x, c_subset)
            c_subset_all_layers[:, :, e] = c_subset.squeeze(-1)

        if return_embedding:
            return x, selected_indices

        return c_subset_all_layers, selected_indices

In [6]:
def load_pretrained_model(checkpoint_path, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Load a pretrained model from a checkpoint file
    
    Args:
        checkpoint_path: Path to the checkpoint file (*_checkpoint.pth)
        device: Device to load the model onto
        
    Returns:
        model: Loaded FluxTransformer model
        config: Model configuration dictionary
        data_info: Data information dictionary
    """
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Extract configuration
    config = checkpoint['config']
    train_losses = checkpoint['train_losses']
    test_losses = checkpoint['test_losses']

    # Initialize model with saved configuration
    model = FluxTransformer(
        vocab_size=config['vocab_size'],
        input_token_indices=config['input_token_indices'],
        d_model=config['d_model'],
        n_heads=config['n_heads'],
        n_layers=config['n_layers'],
        d_ff=config['d_ff'],
        dropout=config['dropout']
    ).to(device)
    
    # Load model weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()  # Set to evaluation mode
    
    print(f"Loaded model from {checkpoint_path}")
    print(f"Configuration: d_model={config['d_model']}, "
          f"n_heads={config['n_heads']}, n_layers={config['n_layers']}, d_ff={config['d_ff']}")
    
    return model, config, train_losses, test_losses, checkpoint['data_info']

### Load model

In [7]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
model_name = "iML1515_MINN_res_500k_d256_h8_l3_ff1024"
checkpoint_path = f"./models/{model_name}/{model_name}_checkpoint.pth"

 # Load the model
model, config, train_losses, test_losses, data_info = load_pretrained_model(checkpoint_path, device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal number of parameters: {total_params:,}")

# Print data information
print("\nData information:")
print(f"Dataset: {data_info['dataset']}")
print(f"Training samples: {data_info['n_train']}")
print(f"Test samples: {data_info['n_test']}")

Loaded model from ./models/iML1515_MINN_res_500k_d256_h8_l3_ff1024/iML1515_MINN_res_500k_d256_h8_l3_ff1024_checkpoint.pth
Configuration: d_model=256, n_heads=8, n_layers=3, d_ff=1024

Total number of parameters: 3,069,729

Data information:
Dataset: ./data/iML1515_MINN_res_training_data_500k_samples.csv
Training samples: 400000
Test samples: 100000


### Load data

In [9]:
# Load data using the same function as during training
#set_seed()

DATA_PATH = f"./data/iML1515_MINN_res_test_data_50000_samples.csv"

#X, y, inputs, outputs, input_token_indices, out_indices = load_data(data_info['dataset'])
X, y, inputs, outputs, input_token_indices, out_indices = load_data(DATA_PATH)

X_train, X_test, y_train, y_test = prepare_tensors(X, y, device=device)

model.eval()

today = date.today().isoformat()
pic_dir = f"./pics/{today}/{model_name}"


Loaded data with 50000 samples from ./data/iML1515_MINN_res_test_data_50000_samples.csv
Extracted input columns count:  27
Extracted output columns count: 2712
Medium constraints are injected into these token indices (in outputs order):
[180, 1525, 106, 40, 45, 92, 104, 164, 304, 621, 700, 859, 913, 970, 1424, 1478, 1485, 1500, 1534, 1535, 1836, 1981, 86, 934, 2243, 2320, 633]
Training samples: 40000
Test samples: 10000


### TabPFN ML-to-flux benchmark

Goncalves-style Table 2 benchmark: transcriptomics + proteomics + fixed glucose/O2 uptake inputs, 45 flux targets, leave-one-out CV, fold-local scaling, and R2/MAE/RMSE/NE reported as mean +/- std across samples.

In [10]:
# TabPFN benchmark in the same style as Goncalves et al. / Tazza et al. Table 2
#
# Settings intentionally mirror the ML2Flux benchmark row used in Table 2:
# - Inputs: transcriptomics + proteomics + measured glucose/O2 uptake fluxes
# - Targets: all fluxomics outputs except the fixed uptake fluxes (45 targets)
# - No omics standard-deviation features
# - Outer CV: leave-one-out over 29 Ishii samples
# - Scaling: StandardScaler fit only on each training fold for X and y
# - Metrics: per-sample R2, MAE, RMSE, NE, then mean +/- std across folds
#
# TabPFNRegressor is single-output, so each flux target is fitted independently
# inside a fold, equivalent in spirit to sklearn MultiOutputRegressor(SVR())
# used for SVM in the Goncalves repository.

import inspect
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from tabpfn import TabPFNRegressor

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

hf_token = os.getenv("HFTOKEN") or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if hf_token:
    os.environ.setdefault("HF_TOKEN", hf_token)
    os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", hf_token)

TABPFN_DATA_DIR = Path("./MINN_data")
TABPFN_FLUXOMICS_FILE = "fluxomics.csv"  # original Goncalves/Ishii signed flux table, not the FBA-fit split table
TABPFN_PRIOR_FLUXES = ["R_EX_glc_e_", "R_EX_o2_e_"]
TABPFN_RANDOM_STATE = 12345
TABPFN_CV_RANDOM_STATE = 12345
TABPFN_DEVICE = "auto"
TABPFN_N_ESTIMATORS = 8  # TabPFN default; no HPO, analogous to default classical ML models in Goncalves
TABPFN_SHOW_TARGET_PROGRESS = False


def _load_goncalves_style_ishii_data(data_dir: Path):
    transcript_path = data_dir / "transcriptomics.csv"
    proteomics_path = data_dir / "proteomics.csv"
    fluxomics_path = data_dir / TABPFN_FLUXOMICS_FILE

    missing = [p for p in [transcript_path, proteomics_path, fluxomics_path] if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing benchmark input files: " + ", ".join(str(p) for p in missing))

    tr = pd.read_csv(transcript_path).set_index("experiment")
    pr = pd.read_csv(proteomics_path).set_index("experiment")
    fl = pd.read_csv(fluxomics_path).set_index("experiment")

    # Preserve the fluxomics/Ishii sample order used for target rows.
    sample_index = fl.index.intersection(tr.index).intersection(pr.index)
    if len(sample_index) == 0:
        raise RuntimeError("No overlapping experiment IDs between transcriptomics, proteomics, and fluxomics.")
    sample_index = [idx for idx in fl.index if idx in set(sample_index)]

    x_df = pd.concat(
        [
            tr.loc[sample_index].add_prefix("transcriptomics_"),
            pr.loc[sample_index].add_prefix("proteomics_"),
        ],
        axis=1,
    )
    y_df = fl.loc[sample_index].copy()

    missing_prior = [c for c in TABPFN_PRIOR_FLUXES if c not in y_df.columns]
    if missing_prior:
        raise ValueError("Missing fixed uptake flux columns in fluxomics.csv: " + ", ".join(missing_prior))

    # ML2Flux/Goncalves fixed-uptake setup: append measured uptake fluxes to X and remove them from y.
    x_df = pd.concat([x_df, y_df[TABPFN_PRIOR_FLUXES]], axis=1)
    y_df = y_df.drop(columns=TABPFN_PRIOR_FLUXES)

    return x_df, y_df


def _make_tabpfn_regressor(seed: int):
    params = inspect.signature(TabPFNRegressor).parameters
    kwargs = {}
    if "device" in params:
        kwargs["device"] = TABPFN_DEVICE
    if "n_estimators" in params:
        kwargs["n_estimators"] = TABPFN_N_ESTIMATORS
    if "show_progress" in params:
        kwargs["show_progress"] = False
    if "random_state" in params:
        kwargs["random_state"] = seed
    elif "seed" in params:
        kwargs["seed"] = seed
    return TabPFNRegressor(**kwargs)


def _per_sample_r2_corr_squared(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    out = np.zeros(y_true.shape[0], dtype=np.float64)
    for i in range(y_true.shape[0]):
        yt = y_true[i]
        yp = y_pred[i]
        if np.std(yt) == 0.0 or np.std(yp) == 0.0:
            out[i] = 0.0
            continue
        r = np.corrcoef(yt, yp)[0, 1]
        out[i] = 0.0 if np.isnan(r) else float(r * r)
    return out


def _per_sample_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    diff = y_true - y_pred
    denom = np.linalg.norm(y_true, axis=1)
    return {
        "R2": _per_sample_r2_corr_squared(y_true, y_pred),
        "MAE": np.mean(np.abs(diff), axis=1),
        "RMSE": np.sqrt(np.mean(diff ** 2, axis=1)),
        "NE": np.nan_to_num(np.linalg.norm(diff, axis=1) / denom, posinf=0.0, neginf=0.0, nan=0.0),
    }


def _format_mean_std(avg, std):
    return f"{avg:.3f} +/- {std:.3f}"


X_tabpfn_df, y_tabpfn_df = _load_goncalves_style_ishii_data(TABPFN_DATA_DIR)
X_tabpfn = X_tabpfn_df.to_numpy(dtype=np.float32)
y_tabpfn = y_tabpfn_df.to_numpy(dtype=np.float32)

n_samples, n_features = X_tabpfn.shape
n_targets = y_tabpfn.shape[1]

if n_samples < 2:
    raise ValueError("Need at least two samples for leave-one-out CV.")

print("=== TabPFN Goncalves-style ML-to-flux benchmark ===")
print(f"Samples: {n_samples}")
print(f"Features: {n_features} (transcriptomics + proteomics + fixed uptake fluxes)")
print(f"Targets: {n_targets} fluxes (fixed uptake fluxes removed)")
print(f"Fixed uptake features: {TABPFN_PRIOR_FLUXES}")
print(f"Fluxomics file: {TABPFN_DATA_DIR / TABPFN_FLUXOMICS_FILE}")
print(f"TabPFN n_estimators: {TABPFN_N_ESTIMATORS}, device: {TABPFN_DEVICE}")

kfold = KFold(n_splits=n_samples, shuffle=True, random_state=TABPFN_CV_RANDOM_STATE)

tabpfn_predictions_matrix = []
tabpfn_fold_rows = []
tabpfn_oof_pred = np.zeros_like(y_tabpfn, dtype=np.float32)
tabpfn_oof_true = y_tabpfn.copy()

start_time = time.time()

for fold_no, (train_idx, test_idx) in enumerate(kfold.split(X_tabpfn, y_tabpfn), start=1):
    sc_X = StandardScaler()
    sc_y = StandardScaler()

    X_train_tabpfn_fold = sc_X.fit_transform(X_tabpfn[train_idx]).astype(np.float32)
    y_train_tabpfn_fold = sc_y.fit_transform(y_tabpfn[train_idx]).astype(np.float32)
    X_test_tabpfn_fold = sc_X.transform(X_tabpfn[test_idx]).astype(np.float32)
    y_test_tabpfn_fold = sc_y.transform(y_tabpfn[test_idx]).astype(np.float32)

    y_pred_scaled = np.zeros_like(y_test_tabpfn_fold, dtype=np.float32)

    print("------------------------------------------------------------------------")
    print(f"[TabPFN] Training for LOO fold {fold_no}/{n_samples} | test={X_tabpfn_df.index[test_idx[0]]}")

    for target_idx, target_name in enumerate(y_tabpfn_df.columns):
        y_train_target = y_train_tabpfn_fold[:, target_idx]

        # TabPFN can be brittle with perfectly constant training targets; this is the exact constant predictor.
        if np.allclose(y_train_target, y_train_target[0]):
            y_pred_scaled[:, target_idx] = y_train_target[0]
            continue

        reg = _make_tabpfn_regressor(seed=TABPFN_RANDOM_STATE)
        reg.fit(X_train_tabpfn_fold, y_train_target)
        pred = np.asarray(reg.predict(X_test_tabpfn_fold), dtype=np.float32).reshape(-1)
        if pred.shape[0] != X_test_tabpfn_fold.shape[0]:
            raise RuntimeError(
                f"Unexpected TabPFN prediction shape for target {target_name}: {pred.shape}, expected ({X_test_tabpfn_fold.shape[0]},)"
            )
        y_pred_scaled[:, target_idx] = pred

        if TABPFN_SHOW_TARGET_PROGRESS and (target_idx + 1) % 10 == 0:
            print(f"  fitted {target_idx + 1}/{n_targets} targets")

    y_pred = sc_y.inverse_transform(y_pred_scaled).astype(np.float32)
    y_true = sc_y.inverse_transform(y_test_tabpfn_fold).astype(np.float32)

    tabpfn_oof_pred[test_idx] = y_pred
    tabpfn_predictions_matrix.append((int(test_idx[0]), y_pred[0].copy()))

    fold_metrics = _per_sample_metrics(y_true, y_pred)
    row = {
        "fold": int(fold_no),
        "sample_index": int(test_idx[0]),
        "experiment": str(X_tabpfn_df.index[test_idx[0]]),
        "R2": float(fold_metrics["R2"][0]),
        "MAE": float(fold_metrics["MAE"][0]),
        "RMSE": float(fold_metrics["RMSE"][0]),
        "NE": float(fold_metrics["NE"][0]),
    }
    tabpfn_fold_rows.append(row)
    print(
        f"Score for fold {fold_no}: "
        f"R2={row['R2']:.4f}; MAE={row['MAE']:.6f}; "
        f"RMSE={row['RMSE']:.6f}; NE={row['NE']:.6f}"
    )

tabpfn_fold_metrics_df = pd.DataFrame(tabpfn_fold_rows)

metric_order = ["R2", "MAE", "RMSE", "NE"]
tabpfn_summary_numeric_df = pd.DataFrame(
    {
        "avg": [float(tabpfn_fold_metrics_df[m].mean()) for m in metric_order],
        "std": [float(tabpfn_fold_metrics_df[m].std(ddof=0)) for m in metric_order],
    },
    index=metric_order,
)

tabpfn_table2_row_df = pd.DataFrame(
    [
        {
            "Model": "TabPFN",
            **{
                m: _format_mean_std(
                    float(tabpfn_summary_numeric_df.loc[m, "avg"]),
                    float(tabpfn_summary_numeric_df.loc[m, "std"]),
                )
                for m in metric_order
            },
        }
    ]
)

tabpfn_predictions_matrix.sort(key=lambda x: x[0])
tabpfn_predictions_df = pd.DataFrame(
    [x[1] for x in tabpfn_predictions_matrix],
    index=X_tabpfn_df.index,
    columns=y_tabpfn_df.columns,
).T

tabpfn_oof_true_df = pd.DataFrame(tabpfn_oof_true, index=X_tabpfn_df.index, columns=y_tabpfn_df.columns)
tabpfn_oof_pred_df = pd.DataFrame(tabpfn_oof_pred, index=X_tabpfn_df.index, columns=y_tabpfn_df.columns)

elapsed_min = (time.time() - start_time) / 60.0

print("------------------------------------------------------------------------")
print(f"TabPFN benchmark finished in {elapsed_min:.1f} min")
print("=== TabPFN Table 2-style row ===")
display(tabpfn_table2_row_df)
print("=== Numeric summary ===")
display(tabpfn_summary_numeric_df)
print("=== Fold metrics ===")
display(tabpfn_fold_metrics_df)

# Keep tabpfn_table2_row_df available for the final combined Table 2-style comparison.


=== TabPFN Goncalves-style ML-to-flux benchmark ===
Samples: 29
Features: 141 (transcriptomics + proteomics + fixed uptake fluxes)
Targets: 45 fluxes (fixed uptake fluxes removed)
Fixed uptake features: ['R_EX_glc_e_', 'R_EX_o2_e_']
Fluxomics file: MINN_data\fluxomics.csv
TabPFN n_estimators: 8, device: auto
------------------------------------------------------------------------
[TabPFN] Training for LOO fold 1/29 | test=tktA
Score for fold 1: R2=0.9729; MAE=0.301595; RMSE=0.580070; NE=0.148093
------------------------------------------------------------------------
[TabPFN] Training for LOO fold 2/29 | test=gpmB
Score for fold 2: R2=0.9912; MAE=0.103702; RMSE=0.230077; NE=0.085272
------------------------------------------------------------------------
[TabPFN] Training for LOO fold 3/29 | test=zwf
Score for fold 3: R2=0.9808; MAE=0.245592; RMSE=0.339671; NE=0.129790
------------------------------------------------------------------------
[TabPFN] Training for LOO fold 4/29 | test=WT

,Model,R2,MAE,RMSE,NE
0,TabPFN,0.969 +/- 0.034,0.369 +/- 0.594,0.562 +/- 0.774,0.197 +/- 0.172


=== Numeric summary ===


,avg,std
R2,0.969187,0.034149
MAE,0.368767,0.594040
RMSE,0.561861,0.774108
NE,0.196814,0.171549


=== Fold metrics ===


,fold,sample_index,experiment,R2,MAE,RMSE,NE
0,1,25,tktA,0.972930,0.301595,0.580070,0.148093
1,2,15,gpmB,0.991164,0.103702,0.230077,0.085272
2,3,19,zwf,0.980823,0.245592,0.339671,0.129790
3,4,3,WT_0.5h-1,0.962955,0.930261,1.309730,0.278696
4,5,12,fbaB,0.967891,0.113820,0.352343,0.183031
5,6,0,REF,0.989457,0.077057,0.208225,0.096288
6,7,23,rpiA,0.983964,0.183911,0.298646,0.123428
7,8,8,pgi,0.834433,0.490874,0.765428,0.460426
8,9,20,pgl,0.990446,0.130694,0.232109,0.088275
9,10,7,pgm,0.965641,0.115923,0.419642,0.173666


### Goncalves pFBA baseline recomputed with iML1515

This cell reruns the pFBA baseline protocol from the Goncalves/omics2flux Ishii benchmark, but uses the iML1515 GEM instead of the original iAF1260 model.

For each Ishii sample, the measured glucose and oxygen uptake values from `MINN_data/fluxomics.csv` are fixed as exchange bounds. For knockout samples, the notebook uses the same ordered b-number knockout list as `omics2flux/pfba.py`; this matters because the local fluxomics table uses gene-symbol sample labels while the Goncalves pFBA script uses b-numbers. Predictions are then extracted for the same 45 non-uptake flux targets used in the Goncalves Table 2 pFBA row, with reaction-name adjustments where iML1515 uses different exchange or biomass IDs.

The output is a Table 2-style pFBA row with R2, MAE, RMSE, and NE reported as mean +/- standard deviation across the 29 Ishii samples, plus per-sample diagnostics and any failed iML1515 pFBA cases.

In [11]:
# Goncalves-style pFBA baseline, reproduced on iML1515 instead of iAF1260
#
# Matches omics2flux/pfba.py conceptually:
# - Ishii sample order from fluxomics.csv
# - fixed glucose and oxygen uptake bounds from measured fluxomics values
# - gene knockouts for b-number samples
# - pFBA predictions for the same 45 non-uptake target fluxes
# - R2, MAE, RMSE, NE as mean +/- std across samples

import logging
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from cobra.io import read_sbml_model
    from cobra.flux_analysis import pfba as cobra_pfba
except ImportError as e:
    raise ImportError("cobra is required for the iML1515 pFBA benchmark. Install cobra in the notebook environment.") from e

logging.getLogger("cobra").setLevel(logging.ERROR)

GONCALVES_PFBA_DATA_DIR = Path("./MINN_data")
GONCALVES_PFBA_FLUXOMICS_FILE = "fluxomics.csv"
GONCALVES_PFBA_MODEL_PATH = Path("./models/iML1515.xml")
GONCALVES_PFBA_OBJECTIVE = globals().get("PFBA_OBJECTIVE_RXN", "BIOMASS_Ec_iML1515_core_75p37M")
GONCALVES_PFBA_WT_SAMPLES = {"REF", "WT_0.1h-1", "WT_0.4h-1", "WT_0.5h-1", "WT_0.7h-1"}
GONCALVES_PFBA_SAMPLE_TO_B_NUMBER = {
    "galM": "b0756",
    "glk": "b2388",
    "pgm": "b0688",
    "pgi": "b4025",
    "pfkA": "b3916",
    "pfkB": "b1723",
    "fbp": "b4232",
    "fbaB": "b2097",
    "gapC": "b0118",
    "gpmA": "b0755",
    "gpmB": "b4395",
    "pykA": "b1854",
    "pykF": "b1676",
    "ppsA": "b1702",
    "zwf": "b1852",
    "pgl": "b0767",
    "gnd": "b2029",
    "rpe": "b3386",
    "rpiA": "b2914",
    "rpiB": "b4090",
    "tktA": "b2935",
    "tktB": "b2465",
    "talA": "b2464",
    "talB": "b0008",
}
# iML1515 does not contain b4395/ytjC from the original iAF1260 pFBA settings.
# Use the iML1515 PGM isozyme gene for the gpmB sample so the adapted benchmark has 29/29 samples.
GONCALVES_PFBA_iML1515_GENE_ALIASES = {"b4395": "b3612"}

GONCALVES_PFBA_TARGETS = [
    "GLCptspp", "PGI", "PFK", "FBA", "TPI", "PGK", "GAPD", "ENO", "PGM", "PYK",
    "PDH", "G6PDH2r", "PGL", "GND", "RPE", "RPI", "TKT1", "TALA", "TKT2", "CS",
    "ACONTb", "ACONTa", "ICDHyr", "SUCOAS", "AKGDH", "SUCDi", "FUM", "MDH", "PPC",
    "ME2", "ICL", "MALS", "ACKr", "PTAr", "LDH_D", "ACALD", "ALCD2x",
    "EX_co2_e_", "EX_etoh_e_", "EX_ac_e_", "EX_lac_D_e_", "EX_succ_e_", "EX_pyr_e_", "EX_for_e_",
    "Ec_biomass_iAF1260_core_59p81M",
]

GONCALVES_TO_iML1515_REACTION = {
    "EX_co2_e_": "EX_co2_e",
    "EX_etoh_e_": "EX_etoh_e",
    "EX_ac_e_": "EX_ac_e",
    "EX_lac_D_e_": "EX_lac__D_e",
    "EX_succ_e_": "EX_succ_e",
    "EX_pyr_e_": "EX_pyr_e",
    "EX_for_e_": "EX_for_e",
    "Ec_biomass_iAF1260_core_59p81M": GONCALVES_PFBA_OBJECTIVE,
}


def _goncalves_target_to_iML1515_reaction(target_name: str) -> str:
    return GONCALVES_TO_iML1515_REACTION.get(target_name, target_name)


def _get_model_gene(model_obj, gene_id: str):
    aliased_gene_id = GONCALVES_PFBA_iML1515_GENE_ALIASES.get(gene_id, gene_id)
    for candidate in (aliased_gene_id, f"G_{aliased_gene_id}"):
        try:
            return model_obj.genes.get_by_id(candidate)
        except KeyError:
            pass
    return None


def _goncalves_r2_metric(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    out = np.zeros(y_true.shape[0], dtype=np.float64)
    for i in range(y_true.shape[0]):
        yt = y_true[i]
        yp = y_pred[i]
        if np.std(yt) == 0.0 or np.std(yp) == 0.0:
            out[i] = 0.0
            continue
        r = np.corrcoef(yt, yp)[0, 1]
        out[i] = 0.0 if np.isnan(r) else float(r * r)
    return out


def _goncalves_sample_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    diff = y_true - y_pred
    denom = np.linalg.norm(y_true, axis=1)
    return {
        "R2": _goncalves_r2_metric(y_true, y_pred),
        "MAE": np.mean(np.abs(diff), axis=1),
        "RMSE": np.sqrt(np.mean(diff ** 2, axis=1)),
        "NE": np.nan_to_num(np.linalg.norm(diff, axis=1) / denom, posinf=0.0, neginf=0.0, nan=0.0),
    }


def _format_mean_std_local(avg, std):
    if "_format_mean_std" in globals():
        return _format_mean_std(avg, std)
    return f"{avg:.3f} +/- {std:.3f}"


fluxomics_path = GONCALVES_PFBA_DATA_DIR / GONCALVES_PFBA_FLUXOMICS_FILE
if not fluxomics_path.exists():
    raise FileNotFoundError(f"Missing fluxomics file: {fluxomics_path}")
if not GONCALVES_PFBA_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing iML1515 SBML model: {GONCALVES_PFBA_MODEL_PATH}")

goncalves_pfba_fluxomics_df = pd.read_csv(fluxomics_path).set_index("experiment")
true_cols = [f"R_{c}" for c in GONCALVES_PFBA_TARGETS]
missing_true_cols = [c for c in true_cols if c not in goncalves_pfba_fluxomics_df.columns]
if missing_true_cols:
    raise ValueError("Missing Goncalves target columns in fluxomics.csv: " + ", ".join(missing_true_cols))
for c in ["R_EX_glc_e_", "R_EX_o2_e_"]:
    if c not in goncalves_pfba_fluxomics_df.columns:
        raise ValueError(f"Missing measured uptake column in fluxomics.csv: {c}")

goncalves_iML1515_model = read_sbml_model(str(GONCALVES_PFBA_MODEL_PATH))
goncalves_iML1515_model.objective = GONCALVES_PFBA_OBJECTIVE

target_rxn_map = {c: _goncalves_target_to_iML1515_reaction(c) for c in GONCALVES_PFBA_TARGETS}
missing_model_rxns = [rxn for rxn in sorted(set(target_rxn_map.values())) if rxn not in goncalves_iML1515_model.reactions]
missing_model_rxns += [rxn for rxn in ["EX_glc__D_e", "EX_o2_e"] if rxn not in goncalves_iML1515_model.reactions]
if missing_model_rxns:
    raise KeyError("Missing required reactions in iML1515: " + ", ".join(missing_model_rxns))

print("=== Goncalves-style pFBA baseline with iML1515 ===")
print(f"Samples: {len(goncalves_pfba_fluxomics_df)}")
print(f"Targets: {len(GONCALVES_PFBA_TARGETS)} fluxes")
print(f"Model: {GONCALVES_PFBA_MODEL_PATH}")
print(f"Objective: {GONCALVES_PFBA_OBJECTIVE}")
print("Fixed uptake bounds: R_EX_glc_e_ -> EX_glc__D_e, R_EX_o2_e_ -> EX_o2_e")
goncalves_pfba_ko_mapping_df = pd.DataFrame(
    [
        {
            "experiment": exp,
            "goncalves_b_number": bnum,
            "iML1515_gene_used": GONCALVES_PFBA_iML1515_GENE_ALIASES.get(bnum, bnum),
        }
        for exp, bnum in GONCALVES_PFBA_SAMPLE_TO_B_NUMBER.items()
    ]
)
print("Goncalves b-number knockout mapping used for non-WT samples:")
display(goncalves_pfba_ko_mapping_df)

prediction_rows = []
failed_rows = []

for experiment, row in goncalves_pfba_fluxomics_df.iterrows():
    glc_bound = float(row["R_EX_glc_e_"])
    o2_bound = float(row["R_EX_o2_e_"])

    with goncalves_iML1515_model as m:
        if experiment not in GONCALVES_PFBA_WT_SAMPLES:
            goncalves_gene_id = GONCALVES_PFBA_SAMPLE_TO_B_NUMBER.get(str(experiment))
            if goncalves_gene_id is None:
                failed_rows.append({"experiment": experiment, "reason": f"missing Goncalves knockout mapping for {experiment}"})
                continue
            gene = _get_model_gene(m, goncalves_gene_id)
            if gene is None:
                failed_rows.append({"experiment": experiment, "reason": f"missing iML1515 gene for Goncalves knockout {goncalves_gene_id}"})
                continue
            gene.knock_out()

        m.reactions.get_by_id("EX_glc__D_e").bounds = (glc_bound, glc_bound)
        m.reactions.get_by_id("EX_o2_e").bounds = (o2_bound, o2_bound)

        try:
            solution = cobra_pfba(m)
            sol_status = getattr(solution, "status", "optimal")
            if sol_status != "optimal":
                raise RuntimeError(f"pFBA status={sol_status}")
            prediction_rows.append(
                {
                    "experiment": experiment,
                    **{target: float(solution.fluxes[target_rxn_map[target]]) for target in GONCALVES_PFBA_TARGETS},
                }
            )
        except Exception as e:
            failed_rows.append({"experiment": experiment, "reason": str(e)})

if not prediction_rows:
    raise RuntimeError("No successful iML1515 pFBA samples. Check uptake bounds, objective, and knockout feasibility.")

goncalves_iML1515_pfba_predictions_df = pd.DataFrame(prediction_rows).set_index("experiment")
goncalves_iML1515_pfba_true_df = goncalves_pfba_fluxomics_df.loc[
    goncalves_iML1515_pfba_predictions_df.index,
    true_cols,
].copy()
goncalves_iML1515_pfba_true_df.columns = GONCALVES_PFBA_TARGETS

y_true_pfba_iML1515 = goncalves_iML1515_pfba_true_df.to_numpy(dtype=np.float64)
y_pred_pfba_iML1515 = goncalves_iML1515_pfba_predictions_df[GONCALVES_PFBA_TARGETS].to_numpy(dtype=np.float64)
pfba_iML1515_metrics = _goncalves_sample_metrics(y_true_pfba_iML1515, y_pred_pfba_iML1515)

metric_order = ["R2", "MAE", "RMSE", "NE"]
goncalves_iML1515_pfba_fold_metrics_df = pd.DataFrame(
    {
        "experiment": goncalves_iML1515_pfba_predictions_df.index,
        **{m: pfba_iML1515_metrics[m] for m in metric_order},
    }
)

goncalves_iML1515_pfba_summary_numeric_df = pd.DataFrame(
    {
        "avg": [float(np.mean(pfba_iML1515_metrics[m])) for m in metric_order],
        "std": [float(np.std(pfba_iML1515_metrics[m])) for m in metric_order],
    },
    index=metric_order,
)

goncalves_iML1515_pfba_table2_row_df = pd.DataFrame(
    [
        {
            "Model": "pFBA iML1515",
            **{
                m: _format_mean_std_local(
                    float(goncalves_iML1515_pfba_summary_numeric_df.loc[m, "avg"]),
                    float(goncalves_iML1515_pfba_summary_numeric_df.loc[m, "std"]),
                )
                for m in metric_order
            },
        }
    ]
)

goncalves_iML1515_pfba_failed_df = pd.DataFrame(failed_rows)

print(f"Successful pFBA samples: {len(goncalves_iML1515_pfba_predictions_df)}/{len(goncalves_pfba_fluxomics_df)}")
if not goncalves_iML1515_pfba_failed_df.empty:
    print("Failed samples:")
    display(goncalves_iML1515_pfba_failed_df)

print("=== iML1515 pFBA Table 2-style row ===")
display(goncalves_iML1515_pfba_table2_row_df)
print("=== Numeric summary ===")
display(goncalves_iML1515_pfba_summary_numeric_df)
print("=== Per-sample metrics ===")
display(goncalves_iML1515_pfba_fold_metrics_df)



=== Goncalves-style pFBA baseline with iML1515 ===
Samples: 29
Targets: 45 fluxes
Model: models\iML1515.xml
Objective: BIOMASS_Ec_iML1515_core_75p37M
Fixed uptake bounds: R_EX_glc_e_ -> EX_glc__D_e, R_EX_o2_e_ -> EX_o2_e
Goncalves b-number knockout mapping used for non-WT samples:


,experiment,goncalves_b_number,iML1515_gene_used
0,galM,b0756,b0756
1,glk,b2388,b2388
2,pgm,b0688,b0688
3,pgi,b4025,b4025
4,pfkA,b3916,b3916
5,pfkB,b1723,b1723
6,fbp,b4232,b4232
7,fbaB,b2097,b2097
8,gapC,b0118,b0118
9,gpmA,b0755,b0755


Successful pFBA samples: 29/29
=== iML1515 pFBA Table 2-style row ===


,Model,R2,MAE,RMSE,NE
0,pFBA iML1515,0.818 +/- 0.167,0.690 +/- 0.761,1.081 +/- 1.066,0.389 +/- 0.188


=== Numeric summary ===


,avg,std
R2,0.817741,0.166915
MAE,0.689957,0.760822
RMSE,1.080738,1.065808
NE,0.389118,0.188365


=== Per-sample metrics ===


,experiment,R2,MAE,RMSE,NE
0,REF,0.854697,0.504620,0.780408,0.360880
1,WT_0.1h-1,0.489591,0.475834,0.732902,0.792481
2,WT_0.4h-1,0.914258,0.658725,0.973097,0.280574
3,WT_0.5h-1,0.920280,0.786240,1.234688,0.262728
4,WT_0.7h-1,0.595764,4.374313,6.253833,0.640742
5,galM,0.936769,0.367946,0.577618,0.249931
6,glk,0.742725,0.810760,1.236964,0.482470
7,pgm,0.839266,0.582875,0.922983,0.381969
8,pgi,0.627307,0.841535,1.133924,0.682086
9,pfkA,0.881120,0.221903,0.701320,0.359021


### Table 2 comparison metrics

This final cell combines the published Tazza et al. Table 2 rows with the reproduced iML1515 pFBA row and the TabPFN row computed above.


In [12]:
# Final Table 2-style comparison metrics.
# Run the TabPFN and iML1515 pFBA benchmark cells above before this cell.

required_table2_rows = [
    "goncalves_iML1515_pfba_table2_row_df",
    "tabpfn_table2_row_df",
]
missing_table2_rows = [name for name in required_table2_rows if name not in globals()]
if missing_table2_rows:
    raise RuntimeError(
        "Run the benchmark cells above before creating the final Table 2 comparison. Missing: "
        + ", ".join(missing_table2_rows)
    )

# Published Tazza et al. Table 2 rows, kept in the original paper order.
tazza_table2_published_rows_df = pd.DataFrame(
    [
        {"Model": "pFBA*", "R2": "0.823 +/- 0.156", "MAE": "0.692 +/- 0.733", "RMSE": "1.058 +/- 1.029", "NE": "0.381 +/- 0.185"},
        {"Model": "NN*", "R2": "0.967 +/- 0.036", "MAE": "0.652 +/- 0.945", "RMSE": "0.936 +/- 1.314", "NE": "0.338 +/- 0.338"},
        {"Model": "RF*", "R2": "0.970 +/- 0.037", "MAE": "0.507 +/- 0.804", "RMSE": "0.729 +/- 1.105", "NE": "0.271 +/- 0.347"},
        {"Model": "MINN-MSE-base", "R2": "0.950 +/- 0.060", "MAE": "0.525 +/- 0.525", "RMSE": "0.736 +/- 0.703", "NE": "0.287 +/- 0.282"},
        {"Model": "MINN-unbalanced", "R2": "0.951 +/- 0.051", "MAE": "0.563 +/- 0.739", "RMSE": "0.814 +/- 1.067", "NE": "0.325 +/- 0.442"},
        {"Model": "MINN-c-balanced", "R2": "0.950 +/- 0.055", "MAE": "0.473 +/- 0.480", "RMSE": "0.678 +/- 0.653", "NE": "0.272 +/- 0.280"},
    ]
)

# Keep reproduced rows after the published rows, with TabPFN last.
goncalves_iML1515_table2_comparison_df = pd.concat(
    [
        tazza_table2_published_rows_df,
        goncalves_iML1515_pfba_table2_row_df,
        tabpfn_table2_row_df,
    ],
    ignore_index=True,
)

print("=== Table 2-style comparison including iML1515 pFBA and TabPFN ===")
display(goncalves_iML1515_table2_comparison_df)



=== Table 2-style comparison including iML1515 pFBA and TabPFN ===


,Model,R2,MAE,RMSE,NE
0,pFBA*,0.823 +/- 0.156,0.692 +/- 0.733,1.058 +/- 1.029,0.381 +/- 0.185
1,NN*,0.967 +/- 0.036,0.652 +/- 0.945,0.936 +/- 1.314,0.338 +/- 0.338
2,RF*,0.970 +/- 0.037,0.507 +/- 0.804,0.729 +/- 1.105,0.271 +/- 0.347
3,MINN-MSE-base,0.950 +/- 0.060,0.525 +/- 0.525,0.736 +/- 0.703,0.287 +/- 0.282
4,MINN-unbalanced,0.951 +/- 0.051,0.563 +/- 0.739,0.814 +/- 1.067,0.325 +/- 0.442
5,MINN-c-balanced,0.950 +/- 0.055,0.473 +/- 0.480,0.678 +/- 0.653,0.272 +/- 0.280
6,pFBA iML1515,0.818 +/- 0.167,0.690 +/- 0.761,1.081 +/- 1.066,0.389 +/- 0.188
7,TabPFN,0.969 +/- 0.034,0.369 +/- 0.594,0.562 +/- 0.774,0.197 +/- 0.172
